# Démo 4 · Valider et régler les modèles

Dans la démo 1, nous avons évalué un modèle avec **une seule** séparation
entraînement/validation. Ce notebook pose deux questions qui reviennent dans tout projet
de machine learning :

1. *Mon modèle est-il vraiment bon ?* Une seule séparation peut être chanceuse ou malchanceuse.
2. *Quels réglages choisir ?* La plupart des modèles ont des **hyperparamètres** (des valeurs
   que l’on fixe avant l’entraînement, comme le nombre de voisins dans k-NN), et il nous faut
   une façon équitable de les choisir.

Cette démo est conçue pour être lue à votre rythme. Exécutez chaque cellule, lisez le
résultat et faites l’exercice **À vous** avant d’ouvrir sa réponse de référence.

À la fin, vous serez capables de :

- Expliquer pourquoi la validation croisée à K plis donne une estimation plus fiable qu’une seule séparation, et quand utiliser sa version stratifiée.
- Utiliser `cross_val_score` pour comparer des modèles.
- Régler des hyperparamètres avec `GridSearchCV` et `RandomizedSearchCV`, et interpréter leurs résultats.
- Enchaîner le prétraitement et un modèle dans un `Pipeline` pour que le réglage ne voie jamais les données de test.

**La règle que nous suivons tout du long :** mettre de côté un ensemble de test → choisir et
régler le modèle par validation croisée sur l’ensemble d’entraînement seulement → évaluer
**une seule fois** sur l’ensemble de test.

Cette démo comprend trois notebooks : **01 (celui-ci)**, puis
[02 · Intervalles de confiance par bootstrap](02_bootstrap_confidence_intervals.ipynb), puis
[03 · Titanic, de bout en bout](03_titanic_end_to_end.ipynb).

## Configuration : choisir l’environnement local ou Colab

**En local :** lancez `uv sync` à la racine du dépôt `ift3700-6758`, puis sélectionnez le noyau de l’environnement commun `.venv` sous Python 3.11. Consultez le [README du dépôt](../../../README.md) pour la configuration. Sautez la cellule d’installation facultative : elle n’installe rien en local.

**Google Colab | sans clonage du dépôt ni installation manuelle de uv :**

1. Ouvrez ce notebook dans Colab et choisissez un runtime **CPU**. L’installation nécessite Internet.
2. Dans la **cellule facultative ci-dessous**, définissez `INSTALL_COLAB_PACKAGES = True` et exécutez-la une fois. La cellule installe `uv`, puis l’utilise pour installer les bibliothèques dans le Python actuel du notebook.
3. Si Colab demande un redémarrage après l’installation, choisissez **Runtime → Restart session**.
4. Remettez le paramètre à `False`, puis exécutez les cellules de haut en bas avec **Maj+Entrée**.

Ce notebook utilise **NumPy** et **pandas** pour les tableaux et les tables, **matplotlib** pour les graphiques, **scikit-learn** pour les modèles et **SciPy** pour les distributions de la recherche aléatoire. Les données (des chiffres manuscrits) sont fournies avec scikit-learn : aucun téléchargement n’est nécessaire.

In [ ]:
# OPTIONAL: run once in a fresh Colab runtime; skip during local development.
INSTALL_COLAB_PACKAGES = False  # Set to True to install; reset to False afterward.

import sys
import subprocess

# Detect Colab; otherwise use the local .venv.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    print("Local environment: skipped. Use uv sync in your terminal.")
elif not INSTALL_COLAB_PACKAGES:
    print("Installation skipped. Set INSTALL_COLAB_PACKAGES = True if this is a fresh Colab runtime.")
else:
    # Install into the Python running these cells.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])
    packages = [
        "numpy>=1.26",
        "pandas>=2.2,<4",
        "matplotlib>=3.7",
        "scikit-learn>=1.4",
        "scipy>=1.11",
    ]
    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install",
                           "--python", sys.executable, *packages])
    print("Installation finished. Restart the session if Colab requests it.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import loguniform

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

## A. Les données : des chiffres manuscrits

Chaque image fait 8×8 pixels, avec des niveaux de gris de 0 à 16. scikit-learn attend une
table avec une ligne par exemple et une colonne par variable : chaque image est donc
aplatie en **64 variables**. La cible `y` est le chiffre (de 0 à 9).

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target
print("X shape:", X.shape, "| y shape:", y.shape)

fig, axes = plt.subplots(1, 8, figsize=(10, 1.8))
for ax, image, label in zip(axes, digits.images, digits.target):
    ax.imshow(image, cmap="gray_r")
    ax.set_title(label)
    ax.axis("off")
plt.show()

## B. Pourquoi une seule séparation ne suffit pas

Entraînons le même arbre de décision cinq fois en changeant seulement **les lignes** qui vont
dans l’ensemble de validation (`random_state`). Le modèle et les données sont identiques ;
seule la séparation change.

In [ ]:
for seed in range(5):
    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.3, random_state=seed)
    tree = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)
    print(f"split {seed}: validation accuracy = {tree.score(X_val, y_val):.3f}")

L’accuracy varie d’environ deux points de pourcentage d’une séparation à l’autre. Si deux
modèles différaient d’autant sur une seule séparation, nous ne pourrions pas savoir si l’un
est vraiment meilleur ou s’il a simplement bénéficié d’une séparation plus favorable. La
**validation croisée** fait plutôt la moyenne sur plusieurs séparations.

## C. Validation croisée à K plis

La validation croisée à K plis découpe les données d’entraînement en *K* parties (les
**plis**, *folds* en anglais). Chaque pli sert à son tour d’ensemble de validation pendant que
le modèle s’entraîne sur les *K − 1* autres plis. On obtient *K* scores, dont on rapporte la
moyenne et la dispersion.

![Validation croisée à 5 plis (documentation de scikit-learn)](https://scikit-learn.org/stable/_images/grid_search_cross_validation.png)

Le schéma illustre aussi la règle de l’introduction : les **données de test** restent en
dehors de la boucle et ne servent qu’à l’évaluation finale.

### Un tout petit exemple pour voir les plis
Dix exemples répartis en trois classes déséquilibrées : deux `a`, deux `b`, six `c`. Les
lignes sont **triées par classe**, ce qui arrive souvent dans les vrais fichiers.

In [ ]:
X_toy = np.arange(10).reshape(-1, 1)
y_toy = np.array(["a", "a", "b", "b", "c", "c", "c", "c", "c", "c"])

k_fold = KFold(n_splits=2, shuffle=False)
for fold, (train_idx, test_idx) in enumerate(k_fold.split(X_toy, y_toy), start=1):
    print(f"fold {fold}: train labels {y_toy[train_idx]} | validation labels {y_toy[test_idx]}")

Au pli 1, le modèle s’entraînerait **uniquement sur des `c`** et devrait reconnaître des `a`
et des `b`, qu’il n’a jamais vus. La validation à K plis simple ne tient pas compte des
étiquettes.

### K plis stratifiés
`StratifiedKFold` construit des plis qui conservent les **proportions des classes** du jeu de
données complet. `shuffle=True` mélange aussi les lignes au préalable ; `random_state` rend
ce mélange reproductible.

In [ ]:
stratified = StratifiedKFold(n_splits=2, shuffle=True, random_state=0)
for fold, (train_idx, test_idx) in enumerate(stratified.split(X_toy, y_toy), start=1):
    print(f"fold {fold}: train labels {y_toy[train_idx]} | validation labels {y_toy[test_idx]}")

Chaque pli contient maintenant toutes les classes, à peu près dans les bonnes proportions.
En classification, les plis stratifiés sont le choix habituel.

### `cross_val_score` : la validation croisée en une ligne
`cross_val_score` exécute toute la boucle à notre place : séparer, entraîner, évaluer,
recommencer. Lorsque vous passez un classifieur et un entier `cv`, scikit-learn utilise
automatiquement des plis stratifiés. Nous passons explicitement un objet `StratifiedKFold`
pour que le mélange soit reproductible.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(DecisionTreeClassifier(random_state=0), X, y, cv=cv)
print("fold scores:", np.round(scores, 3))
print(f"mean = {scores.mean():.3f}, standard deviation = {scores.std():.3f}")

La moyenne est une estimation plus stable que n’importe quelle séparation unique, et
l’écart-type indique à quel point le score dépend des données vues par le modèle.

### À vous
Utilisez `cross_val_score` avec le même `cv` pour comparer `KNeighborsClassifier` avec
`n_neighbors` dans `[1, 3, 5, 9, 15]`. Affichez la moyenne et l’écart-type pour chaque valeur.
Quel *k* choisiriez-vous ?

In [ ]:
# Your turn: loop over the values of k and print the cross-validation scores.

<details>
<summary>Réponse de référence — à ouvrir après avoir essayé</summary>

```python
for k in [1, 3, 5, 9, 15]:
    knn_scores = cross_val_score(KNeighborsClassifier(n_neighbors=k), X, y, cv=cv)
    print(f"k = {k:2d}: {knn_scores.mean():.3f} ± {knn_scores.std():.3f}")
```

Toutes les valeurs de *k* obtiennent environ 0,98 à 0,99, bien au-dessus de l’arbre de
décision. Les scores pour *k* = 1, 3 et 5 sont à environ un écart-type les uns des autres,
tandis que *k* = 15 commence à baisser : n’importe quel petit *k* est donc un choix
raisonnable. Écrire cette boucle à la main, c’est exactement ce que `GridSearchCV`
automatise dans la section suivante.

</details>

## D. Recherche d’hyperparamètres

Un **classifieur à vecteurs de support** (`SVC`) a plusieurs hyperparamètres :

- `kernel` : le noyau, qui détermine la forme de la frontière de décision (`"linear"` ou `"rbf"`, plus flexible).
- `C` : la force avec laquelle les erreurs d’entraînement sont pénalisées. Un `C` élevé colle davantage aux données d’entraînement.
- `gamma` (rbf seulement) : la portée de l’influence d’un exemple d’entraînement.

Nous mettons d’abord de côté **la moitié des données comme ensemble de test**. Tout le réglage
se fait sur l’autre moitié (l’ensemble de *développement*) par validation croisée.

In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.5, stratify=y, random_state=0
)
print("development set:", X_dev.shape, "| test set:", X_test.shape)

### (a) Recherche par grille
`GridSearchCV` essaie **toutes les combinaisons** d’une grille. Pour chaque combinaison, il
effectue une validation croisée sur l’ensemble de développement, puis il réentraîne la
meilleure combinaison sur tout l’ensemble de développement.

Ici, la grille contient 2 × 4 = 8 combinaisons rbf plus 4 combinaisons linéaires :
12 candidats × 5 plis = 60 entraînements. Nous évaluons avec le **macro F1** (la
moyenne du F1 sur les dix classes), qui accorde le même poids à chaque chiffre.

In [ ]:
param_grid = [
    {"kernel": ["rbf"], "gamma": [1e-3, 1e-4], "C": [1, 10, 100, 1000]},
    {"kernel": ["linear"], "C": [1, 10, 100, 1000]},
]

grid_search = GridSearchCV(SVC(), param_grid, scoring="f1_macro", cv=cv)
grid_search.fit(X_dev, y_dev)

print("best parameters:", grid_search.best_params_)
print(f"best cross-validated macro F1: {grid_search.best_score_:.3f}")

`cv_results_` conserve le score de chaque candidat. Une table triée par rang montre si le
gagnant est nettement meilleur ou seulement de peu.

In [ ]:
results = pd.DataFrame(grid_search.cv_results_)
columns = ["param_kernel", "param_C", "param_gamma", "mean_test_score", "std_test_score", "rank_test_score"]
display(results[columns].sort_values("rank_test_score").round(4))

Les trois meilleurs candidats sont à égalité : avec `gamma = 0.001`, tout `C` supérieur ou
égal à 10 donne les mêmes prédictions sur les mêmes plis. De nombreux candidats sont à moins
d’un écart-type du meilleur : sur ces données, le choix exact importe donc peu.

Maintenant, et seulement maintenant, nous évaluons le modèle sélectionné **une seule fois**
sur l’ensemble de test. Le rapport donne la precision, le recall et le F1 score pour chaque chiffre.

In [ ]:
y_pred = grid_search.predict(X_test)
print(classification_report(y_test, y_pred))

### (b) Recherche aléatoire
Une grille grossit vite : 4 hyperparamètres avec 10 valeurs chacun représentent déjà 10 000
candidats. `RandomizedSearchCV` **tire** plutôt un nombre fixe de candidats (`n_iter`) à
partir de distributions que nous décrivons.

`loguniform(1, 1000)` tire des valeurs réparties uniformément sur une **échelle
logarithmique** entre 1 et 1 000, ce qui convient à `C` et à `gamma`, car c’est leur ordre de
grandeur qui compte. Avec le même budget de 12 candidats, la recherche aléatoire explore
beaucoup plus de valeurs distinctes de chaque hyperparamètre que la grille.

In [ ]:
param_distributions = [
    {"kernel": ["rbf"], "gamma": loguniform(1e-4, 1e-3), "C": loguniform(1, 1000)},
    {"kernel": ["linear"], "C": loguniform(1, 1000)},
]

random_search = RandomizedSearchCV(
    SVC(), param_distributions, n_iter=12, scoring="f1_macro", cv=cv, random_state=0
)
random_search.fit(X_dev, y_dev)

print("best parameters:", random_search.best_params_)
print(f"best cross-validated macro F1: {random_search.best_score_:.3f}")
print(f"test macro F1: {f1_score(y_test, random_search.predict(X_test), average='macro'):.3f}")

Les deux recherches atteignent ici à peu près le même score, car le SVC est peu sensible à
ces valeurs sur les chiffres manuscrits. La recherche aléatoire devient rentable lorsqu’il y a
beaucoup d’hyperparamètres et que seuls quelques-uns comptent vraiment.

## E. Pipelines

Les vrais modèles ont généralement besoin d’un **prétraitement** avant l’estimateur : mise à
l’échelle, réduction de dimension, encodage des catégories… Un `Pipeline` enchaîne ces étapes
en un seul objet doté de `fit` et de `predict`. Cela apporte trois avantages :

- **Commodité :** un seul appel à `fit` et un seul à `predict` exécutent toutes les étapes dans l’ordre.
- **Réglage conjoint :** une recherche par grille peut régler le prétraitement et le modèle ensemble.
- **Aucune fuite de données :** les étapes de prétraitement apprennent elles aussi à partir des
  données. `PCA` apprend une moyenne et des directions ; `StandardScaler` apprend des moyennes et des écarts-types.
  Si on les ajuste sur le jeu de données complet avant la validation croisée, les plis de
  validation ont déjà influencé le modèle et les scores sont trop optimistes. Dans un pipeline,
  chaque pli réajuste le prétraitement uniquement sur sa propre partie d’entraînement.

Notre pipeline : conserver les *n* premières composantes principales (PCA) des pixels → régression logistique. Les paramètres d’un pipeline se nomment `step__parameter` (le nom de l’étape, puis celui du paramètre), avec **deux** traits de soulignement.

In [ ]:
pipe = Pipeline(steps=[
    ("pca", PCA()),
    # A large tolerance keeps this example fast.
    ("logistic", LogisticRegression(max_iter=10_000, tol=0.1)),
])

param_grid = {
    "pca__n_components": [5, 15, 30, 45, 64],
    "logistic__C": np.logspace(-4, 4, 4),
}

search = GridSearchCV(pipe, param_grid, cv=cv)
search.fit(X_dev, y_dev)

print("best parameters:", search.best_params_)
print(f"best cross-validated accuracy: {search.best_score_:.3f}")

In [ ]:
print(classification_report(y_test, search.predict(X_test)))

Ce pipeline est moins performant que le SVC réglé plus haut (environ 0,93 contre 0,98) : la
régression logistique trace des frontières linéaires. L’intérêt de cette section, c’est la
mécanique, qui est la même pour n’importe quel modèle.

### Combien de composantes avons-nous conservées ?
Le graphique montre la **variance expliquée cumulée** : la part de la variation des pixels que
préservent les *n* premières composantes. La ligne pointillée indique le nombre choisi par la
recherche.

In [ ]:
best_pca = search.best_estimator_.named_steps["pca"]
pca_full = PCA().fit(X_dev)

plt.plot(np.arange(1, pca_full.n_components_ + 1), np.cumsum(pca_full.explained_variance_ratio_))
plt.axvline(best_pca.n_components, linestyle=":", color="black", label="n_components chosen")
plt.xlabel("Number of PCA components")
plt.ylabel("Cumulative explained variance")
plt.legend()
plt.show()

La recherche n’a pas simplement conservé le plus de variance possible : elle a choisi le
nombre de composantes qui donnait le meilleur score de **validation**.

## À retenir

- Une seule séparation donne une estimation bruitée. La **validation croisée à K plis** fait la moyenne sur *K* séparations ; utilisez des plis **stratifiés** en classification.
- La **recherche par grille** essaie toutes les combinaisons ; la **recherche aléatoire** tire un budget fixe de candidats et s’adapte mieux à un grand nombre d’hyperparamètres.
- Réglez le modèle sur l’ensemble de développement par validation croisée, puis évaluez-le **une seule fois** sur l’ensemble de test.
- Placez le prétraitement dans un **pipeline** pour que chaque pli l’apprenne uniquement à partir des données d’entraînement.

**Suite :** [02 · Intervalles de confiance par bootstrap](02_bootstrap_confidence_intervals.ipynb)
s’intéresse à l’incertitude d’un score.

**Pour aller plus loin :** le guide d’utilisation de scikit-learn sur
[la validation croisée](https://scikit-learn.org/stable/modules/cross_validation.html),
[la recherche d’hyperparamètres](https://scikit-learn.org/stable/modules/grid_search.html) et
[les pipelines](https://scikit-learn.org/stable/modules/compose.html).